# Bitonic Sort Using CUDA
Done by:
Siddharth Sudhakar (25901335)

In [1]:
!pip install numba

In [2]:
import numpy as np
import math
from numba import cuda

In [3]:
@cuda.jit
def bitonic_sort_kernel(arr, step, substep):
    idx = cuda.grid(1)

    n   = arr.shape[0]
    partner   = idx ^ substep
    ascending = (idx & step) == 0

    if partner > idx and partner < n:
        if ascending and arr[idx] > arr[partner]:
            arr[idx], arr[partner] = arr[partner], arr[idx]
        elif not ascending and arr[idx] < arr[partner]:
            arr[idx], arr[partner] = arr[partner], arr[idx]

In [4]:
def bitonic_sort_gpu(arr):
    n = len(arr)
    assert n & (n - 1) == 0, "Array length must be a power of 2"

    d_arr = cuda.to_device(arr.copy())

    threads = 256
    blocks  = math.ceil(n / threads)

    step = 2
    while step <= n:
        substep = step >> 1
        while substep > 0:
            bitonic_sort_kernel[blocks, threads](d_arr, step, substep)
            cuda.synchronize()
            substep >>= 1
        step <<= 1

    return d_arr.copy_to_host()

In [8]:
arr = np.array([5, 3, 8, 1, 9, 2, 7, 4])

In [9]:
print(f"Before: {arr.tolist()}")

Before: [5, 3, 8, 1, 9, 2, 7, 4]


In [10]:
sorted_arr = bitonic_sort_gpu(arr)
print(f"After : {sorted_arr.tolist()}")

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


After : [1, 2, 3, 4, 5, 7, 8, 9]
